# Laboratory 15 — Phase transitions and the Ising model

In this laboratory you build a lattice of spins, watch it order and disorder, and then run the
experiments the module is built on. You check the simulation against the exact solution of the
one-dimensional chain before trusting it with anything else; sweep the square lattice through
its transition for three sizes, reading the susceptibility and the heat capacity out of
fluctuations; measure how long the simulation takes to forget itself, and where that time
peaks; watch a small lattice hop between its two magnetized states while a larger one refuses
to; sweep a field up and down below and above the transition; and run the same physics under
two different update rules. You finish with a measurement of the transition temperature, with
its uncertainty stated honestly.

The route: the objects (Part 1); first look at the lattice (Part 2); the one-dimensional chain
against its exact solution (Part 3); a temperature sweep for three sizes (Part 4); critical
slowing down (Part 5); who chooses the direction (Part 6); hysteresis (Part 7); two update rules,
one equilibrium (Part 8); a measurement of $T_c$ (Part 9); the automated checks (Part 10); and
free exploration (Part 11).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

**Units.** Energies are in units of the coupling $J$, temperatures in units of $J/\kB$ and
fields in units of $J$, so `T = 2.0` means $\kB T = 2J$. The exact transition of the square
lattice is at $2.269$.

## Model specification

| | |
|---|---|
| **System** | an $L \times L$ square lattice of classical spins $s_i = \pm 1$, nearest-neighbour coupling $J > 0$, uniform field $h$; in Part 3, a ring of spins |
| **Dynamics** | a Markov chain, not equations of motion: checkerboard half-sweeps with the Metropolis rule $\min(1, e^{-\Delta E/(\kB T)})$, or where stated the Glauber rule; sweeps are not a physical time |
| **Boundary** | periodic; $T$ and $h$ imposed by an implicit bath |
| **Ensemble** | canonical at $T$, the stationary distribution of the chain by detailed balance |
| **Ignored** | quantum spin, lattice vibrations, long-range dipolar forces, anisotropy, disorder, domain-wall pinning |
| **Valid when** | the chain has run past its burn-in and for many autocorrelation times $\tau(T)$ |
| **Failure modes** | near $T_c$ $\tau$ grows with $L$ and naive error bars are illusory; below $T_c$ at small $|h|$ a run stays in one magnetization branch; a trace that keeps its burn-in is biased |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import ising
from thermolab.validation import relative_error, seed_study

# Every stochastic call below takes an explicit generator; nothing touches global random
# state, so re-running a cell reproduces it exactly.
rng = np.random.default_rng(15)

T_C = ising.onsager_tc()
print(f"Onsager's exact T_c = {T_C:.6f} J/k_B;  mean field says {ising.mean_field_tc():.0f}")

## Part 1 — The objects

| function | what it gives |
|---|---|
| `random_state(L, rng)`, `aligned_state(L)` | a hot (random) or cold (all-up) $L \times L$ lattice; `dimension=1` gives a ring |
| `simulate(state, T, n, rng, burn_in, thin, rule)` | runs the chain: discards `burn_in` sweeps, then records $m$ and $e = E/N$ every `thin` sweeps; returns the two traces and the final state |
| `observables(m, e, T, N)` | $\langle |m| \rangle$ and $\langle e \rangle$ with $\tau$-corrected errors, $\chi' = N\,\mathrm{Var}(|m|)/T$, $c = N\,\mathrm{Var}(e)/T^2$, and $\tau$ of each trace |
| `temperature_scan(L, temps, n, rng)` | `observables` along a ladder of temperatures |
| `autocorrelation_time(trace)`, `mean_with_error(trace)` | integrated $\tau$; mean, standard error from $N_{\mathrm{eff}} = n/(2\tau)$, and $\tau$ |
| `hysteresis_sweep(state, T, fields, sweeps, rng)`, `hysteresis_fields(h_max, n)`, `loop_area` | $m$ along a field ladder $+h \to -h \to +h$, and the loop's area |
| `branch_flips(m)` | how many times a trace crosses between $m > 0.5$ and $m < -0.5$ |
| `ising_1d_exact(T, h)` | the exact $(e, m)$ of the infinite ring |
| `onsager_tc()`, `onsager_magnetization(T)` | the exact square-lattice $T_c$ and spontaneous magnetization |
| `mean_field_magnetization(T, h)` | the solution of $m = \tanh((4m + h)/T)$ |

The state keeps its energy and total spin up to date flip by flip; Part 10 checks that the
bookkeeping never drifts from a recount.

### Predict

Commit to an answer for each before running anything.

1. A $64 \times 64$ lattice at $T = 1.5$, run from a random start and from an all-up start: do the
   two runs end at the same $|m|$? At $T = 3.5$?
2. At which temperature does the simulation take longest to decorrelate: $1.5$, $2.27$ or $3.5$?
3. Is $\langle |m| \rangle(T)$ steeper at the transition for $L = 8$ or for $L = 32$?
4. Below $T_c$ in zero field, does an $8 \times 8$ lattice keep the sign of its magnetization
   over a long run? A $32 \times 32$ lattice?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — First look at the lattice

Run a $64 \times 64$ lattice for 400 sweeps at $T = 1.5$ and at $T = 3.5$, each from a hot start
and from a cold start, and look at the final configurations and the magnetization traces.

In [ ]:
first_look = {}
fig, axes = plt.subplots(2, 4, figsize=(13, 6), gridspec_kw={"height_ratios": [1.0, 0.6]})
for k, (temperature, start) in enumerate([(1.5, "hot"), (1.5, "cold"), (3.5, "hot"),
                                         (3.5, "cold")]):
    state = ising.random_state(64, rng) if start == "hot" else ising.aligned_state(64)
    m, e, state = ising.simulate(state, temperature, 400, rng, burn_in=0)
    first_look[(temperature, start)] = m
    axes[0, k].imshow(state.lattice, cmap="Blues", vmin=-1.5, vmax=1)
    axes[0, k].set_title(f"T = {temperature}, {start} start")
    axes[0, k].set_xticks([])
    axes[0, k].set_yticks([])
    axes[1, k].plot(m, lw=1)
    axes[1, k].set_ylim(-1.05, 1.05)
    axes[1, k].set_xlabel("sweep")
axes[1, 0].set_ylabel("m")
plt.tight_layout()
plt.show()

for (temperature, start), m in first_look.items():
    print(f"T = {temperature}, {start:>4} start: m over the last 100 sweeps = "
          f"{m[-100:].mean():+.3f}")
print(f"Onsager-Yang |m| at T = 1.5: {ising.onsager_magnetization(1.5):.4f}")

At $T = 3.5$ both starts forget where they came from within a few sweeps and wander around
$m = 0$. At $T = 1.5$ the cold start stays near $0.987$; the hot start is a patchwork of domains
that is still coarsening after 400 sweeps — and sometimes gets stuck in two stripes for far
longer. *Neither* is evidence that the equilibrium at $1.5$ is disordered: it is evidence that
reaching it from a random start is slow.

## Part 3 — Calibrate first: the chain against its exact solution

A simulation earns trust by reproducing something known exactly. The ring has an exact
solution (the module's step 3): at $h = 0$, $e = -\tanh(1/T)$ and $m = 0$; in a field,
$m = \sinh(h/T)/\sqrt{\sinh^2(h/T) + e^{-4/T}}$. Run a 1024-site ring under four independent
seeds at each of three points and compare. These runs use the Glauber rule, because in one
dimension at $h = 0$ Metropolis moves domain walls deterministically under the checkerboard and
forgets its starting point slowly — the `ising` module's docstring has the measurement.

### Predict

Will the simulated energies agree with the exact ones to within three standard errors? What
would it mean if they did not?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
chain_points = [(1.0, 0.0), (2.0, 0.0), (1.0, 0.3)]
chain = {}
for temperature, field in chain_points:
    def run(gen, temperature=temperature, field=field):
        state = ising.random_state(1024, gen, field=field, dimension=1)
        m, e, _ = ising.simulate(state, temperature, 500, gen, burn_in=300, rule="glauber")
        return e.mean(), m.mean()

    results = np.array([run(np.random.default_rng(s))
                        for s in np.random.SeedSequence(3).spawn(4)])
    chain[(temperature, field)] = results
    e_exact, m_exact = ising.ising_1d_exact(temperature, field)
    e_se = results[:, 0].std(ddof=1) / 2
    m_se = results[:, 1].std(ddof=1) / 2
    print(f"T = {temperature}, h = {field}:  e = {results[:, 0].mean():.4f} +/- {e_se:.4f} "
          f"(exact {e_exact:.4f}, {abs(results[:, 0].mean() - e_exact) / e_se:.1f} sigma);  "
          f"m = {results[:, 1].mean():+.4f} +/- {m_se:.4f} (exact {m_exact:+.4f})")

t_line = np.linspace(0.3, 4.0, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_line, ising.ising_1d_exact(t_line)[0], color="k", label="exact, h = 0")
ax.plot(t_line, ising.ising_1d_exact(t_line, 0.3)[0], color="grey", ls="--",
        label="exact, h = 0.3")
for (temperature, field), results in chain.items():
    ax.errorbar(temperature, results[:, 0].mean(), yerr=3 * results[:, 0].std(ddof=1) / 2,
                fmt="o", color="#dc2626" if field else "#2563eb", capsize=3)
ax.set_xlabel("k_B T / J")
ax.set_ylabel("E / (N J)")
ax.legend()
plt.tight_layout()
plt.show()

The simulation reproduces the exact chain within its error bars. It has now earned the right to
be asked about the square lattice, which has no exact solution for most of what we will measure.

## Part 4 — A temperature sweep for three sizes

Sweep the square lattice from $T = 3.3$ down to $1.8$ for $L = 8$, $16$ and $32$, with the
temperature grid concentrated near $2.3$. At each temperature, discard 300 sweeps and keep 1200.
From the traces: $\langle |m| \rangle$, the susceptibility $\chi' = N\,\mathrm{Var}(|m|)/T$ and the
heat capacity per spin $c = N\,\mathrm{Var}(e)/T^2$ — module 12's fluctuation identities, with
$|m|$ in place of $m$ for the reason the module's step 6 gives.

This is the laboratory's longest cell: about $10^5$ sweeps in all. In the browser expect a
minute or two.

### Predict

Sketch $\langle|m|\rangle(T)$ for the three sizes on one plot before running. Where will $\chi'$
peak, and will the peak move as $L$ grows?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
SIZES = (8, 16, 32)
temps = np.array([3.3, 3.0, 2.8, 2.65, 2.55, 2.48, 2.42, 2.37, 2.32, 2.28, 2.24, 2.2, 2.1, 1.95,
                  1.8])
scans = {}
for size in SIZES:
    # temperature_scan walks the ladder in the order given, starting each T from the last
    # configuration; cooling from hot avoids starting in an ordered state above T_c.
    scans[size] = ising.temperature_scan(size, temps, 1200, rng, burn_in=300)


def column(size, name):
    return np.array([getattr(r, name) for r in scans[size]])


colours = {8: "#059669", 16: "#d97706", 32: "#dc2626"}
t_fine = np.linspace(1.75, 3.35, 400)
fig, (ax_m, ax_chi, ax_c) = plt.subplots(1, 3, figsize=(14, 4))
ax_m.plot(t_fine, ising.onsager_magnetization(t_fine), color="k", lw=1.2,
          label="Onsager, L = infinity")
ax_m.plot(t_fine, ising.mean_field_magnetization(t_fine), color="grey", ls=":",
          label="mean field")
for size in SIZES:
    ax_m.errorbar(temps, column(size, "abs_magnetization"),
                  yerr=column(size, "abs_magnetization_error"), fmt="o-", ms=3,
                  color=colours[size], label=f"L = {size}")
    ax_chi.plot(temps, column(size, "susceptibility"), "o-", ms=3, color=colours[size])
    ax_c.plot(temps, column(size, "heat_capacity"), "o-", ms=3, color=colours[size])
ax_m.set_ylabel("<|m|>")
ax_chi.set_ylabel("chi' (per spin)")
ax_c.set_ylabel("c (per spin, units of k_B)")
for ax in (ax_m, ax_chi, ax_c):
    ax.axvline(T_C, color="grey", ls="--", lw=1)
    ax.set_xlabel("k_B T / J")
ax_m.legend(fontsize=8)
plt.tight_layout()
plt.show()

peak_t = {size: temps[np.argmax(column(size, "susceptibility"))] for size in SIZES}
for size in SIZES:
    chi = column(size, "susceptibility")
    print(f"L = {size:2d}: chi' peaks at T = {peak_t[size]:.2f} with height {chi.max():6.2f};  "
          f"c peaks at T = {temps[np.argmax(column(size, 'heat_capacity'))]:.2f}")

Three things to read off. The magnetization curves are rounded shoulders, sharper for larger $L$;
none of them touches zero, because on a finite lattice $|m|$ never vanishes. The susceptibility
peak grows with $L$ and moves *down* toward $T_c$. And the mean-field curve is not even in the
right place: it orders at $4.0$.

## Part 5 — Critical slowing down

Every `observables` record also carries the autocorrelation time of its $|m|$ trace, in sweeps.
Plot it against temperature.

### Predict

At which temperature will $\tau$ be largest? Will it grow or shrink with $L$ there?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for size in SIZES:
    ax.plot(temps, column(size, "tau_abs_magnetization"), "o-", ms=4, color=colours[size],
            label=f"L = {size}")
ax.axvline(T_C, color="grey", ls="--", lw=1)
ax.set_yscale("log")
ax.set_xlabel("k_B T / J")
ax.set_ylabel("tau of |m| (sweeps)")
ax.legend()
plt.tight_layout()
plt.show()

for size in SIZES:
    tau = column(size, "tau_abs_magnetization")
    print(f"L = {size:2d}: tau = {tau[0]:.1f} at T = {temps[0]}, {tau.max():.1f} at "
          f"T = {temps[np.argmax(tau)]}, {tau[-1]:.1f} at T = {temps[-1]}")
tau_peak = {size: column(size, "tau_abs_magnetization").max() for size in SIZES}

The simulation is slowest exactly at the transition, and the peak grows with the lattice: that
is critical slowing down. Deep in either phase $|m|$ decorrelates within a sweep or two. Every
error bar in Part 4 already accounts for it — $\sigma/\sqrt{N_{\mathrm{eff}}}$ with
$N_{\mathrm{eff}} = n/(2\tau)$ — which is why the points near $T_c$ carry the largest bars.

## Part 6 — Who chooses the direction?

Below $T_c$ in zero field the canonical average of $m$ is exactly zero, by symmetry. Run an
$8 \times 8$ lattice and a $32 \times 32$ lattice at $T = 2.1$ for 6000 sweeps each, from an
all-up start, and watch the sign of $m$.

### Predict

How many times will each lattice reverse its magnetization? What will the time average of $m$
be for each, and of $|m|$?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
branch = {}
fig, ax = plt.subplots(figsize=(10, 3.5))
for size, colour in ((8, "#059669"), (32, "#dc2626")):
    m, _, _ = ising.simulate(ising.aligned_state(size), 2.1, 6000, rng, burn_in=0)
    branch[size] = m
    ax.plot(m, lw=0.8, color=colour, label=f"L = {size}")
ax.axhline(ising.onsager_magnetization(2.1), color="grey", ls="--", lw=1)
ax.axhline(-ising.onsager_magnetization(2.1), color="grey", ls="--", lw=1)
ax.set_ylim(-1.05, 1.05)
ax.set_xlabel("sweep")
ax.set_ylabel("m")
ax.legend()
plt.tight_layout()
plt.show()

for size, m in branch.items():
    print(f"L = {size:2d}: {ising.branch_flips(m):3d} reversals;  <m> = {m.mean():+.3f};  "
          f"<|m|> = {np.abs(m).mean():.3f}")
print(f"Onsager-Yang m_0(2.1) = {ising.onsager_magnetization(2.1):.4f}")

The small lattice reverses again and again, and its time average drifts toward zero, the
ensemble's answer. The large one never reverses: its time average is $+0.87$, the
spontaneous magnetization, and it would stay there far beyond any run you could afford. Both
have the same $\langle |m| \rangle$. The ensemble average is zero; the *sample* is not — and the
larger the sample, the more completely the two stop being the same thing.

## Part 7 — Hysteresis

Sweep the field $+1 \to -1 \to +1$ in 81 steps, 10 sweeps per step, on a $32 \times 32$ lattice
at $T = 1.5$ (below $T_c$) and at $T = 3.0$ (above). Then repeat the cold loop more slowly.

### Predict

Will either loop close? Which one, if you sweep more slowly?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
fields = ising.hysteresis_fields(1.0, 41)
loops = {}
for temperature, per_step in ((1.5, 10), (1.5, 40), (3.0, 10)):
    h, m, _ = ising.hysteresis_sweep(ising.aligned_state(32), temperature, fields, per_step,
                                     rng)
    loops[(temperature, per_step)] = (h, m, ising.loop_area(h, m))

fig, ax = plt.subplots(figsize=(6, 4.5))
styles = {(1.5, 10): ("#2563eb", "-"), (1.5, 40): ("#2563eb", ":"), (3.0, 10): ("grey", "-")}
for key, (h, m, area) in loops.items():
    colour, ls = styles[key]
    ax.plot(h, m, color=colour, ls=ls, label=f"T = {key[0]}, {key[1]} sweeps/step: area {area:.2f}")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("h / J")
ax.set_ylabel("m")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for key, (h, m, area) in loops.items():
    at_zero = m[np.argmin(np.abs(h[:41]))]
    print(f"T = {key[0]}, {key[1]:2d} sweeps per step: loop area {area:.3f};  "
          f"m at h = 0 on the way down {at_zero:+.2f}")

Below $T_c$ the magnetization keeps its sign well past $h = 0$ and then reverses abruptly: the
lattice is *metastable* — module 14's superheated liquid, in magnetic form — until a reversed
domain large enough to grow happens to form. Sweeping four times more slowly narrows the loop
but does not close it. Above $T_c$ the curve passes through the origin and almost retraces
itself; what remains of its area is lag and noise. Along $h = 0$ below $T_c$ the magnetization
jumps between $\pm m_0$: a first-order line, ending at the critical point.

## Part 8 — Two update rules, one equilibrium

Detailed balance fixes *where* the chain goes and says nothing about how fast. Run Metropolis
and Glauber side by side: first the same quench (random start, $T = 1.5$, $32 \times 32$) under
three seeds each, then the equilibrium energy at $T = 2.8$ under four seeds each.

### Predict

Which rule relaxes faster after the quench? Will they agree on the equilibrium energy at $2.8$?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
relax = {}
for rule in ("metropolis", "glauber"):
    traces = [ising.simulate(ising.random_state(32, np.random.default_rng(s)), 1.5, 150,
                             np.random.default_rng(s), burn_in=0, rule=rule)[1]
              for s in np.random.SeedSequence(8).spawn(3)]
    relax[rule] = np.mean(traces, axis=0)


def equilibrium_energy(rule):
    def measure(gen):
        _, e, _ = ising.simulate(ising.random_state(16, gen), 2.8, 900, gen, burn_in=200,
                                 rule=rule)
        return float(e.mean())
    return seed_study(measure, n_seeds=4)


equilibrium = {rule: equilibrium_energy(rule) for rule in ("metropolis", "glauber")}

fig, ax = plt.subplots(figsize=(7, 4))
for rule, colour in (("metropolis", "#2563eb"), ("glauber", "#dc2626")):
    ax.plot(np.arange(1, 151), relax[rule], color=colour, label=rule)
ax.set_xscale("log")
ax.set_xlabel("sweep")
ax.set_ylabel("E / (N J) after a quench to T = 1.5")
ax.legend()
plt.tight_layout()
plt.show()

for rule in relax:
    study = equilibrium[rule]
    print(f"{rule:>10}: e after 5, 20, 150 sweeps = {relax[rule][[4, 19, 149]].round(3)};  "
          f"equilibrium e at T = 2.8: {study.mean:.4f} +/- {study.standard_error:.4f}")
rule_gap = abs(equilibrium["metropolis"].mean - equilibrium["glauber"].mean)
rule_sigma = np.hypot(equilibrium["metropolis"].standard_error,
                      equilibrium["glauber"].standard_error)
print(f"difference of equilibria: {rule_gap:.4f} = {rule_gap / rule_sigma:.1f} combined sigma")

Glauber relaxes more slowly — it accepts every move less often — and both arrive at the same
equilibrium. A "sweep" is a unit of computation, not of time: no physical constant converts it
into seconds, and Monte Carlo does not show you how a real magnet moves.

## Part 9 — A measurement: the transition temperature

Locate the susceptibility peak for each size more precisely than the grid of Part 4 allows, by
fitting a parabola through the three highest points, and extrapolate the peak positions to
$L \to \infty$ assuming $T_{\mathrm{peak}}(L) = T_c + a/L$ (finite-size scaling with the exact
exponent $\nu = 1$, quoted, not derived). Then check the $L = 16$ peak under three further seeds.

### Predict

Will the extrapolated $T_c$ land within $0.05$ of Onsager's $2.269$? Will the four $L = 16$
peak positions agree to better than $0.05$?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
def parabola_peak(t, y):
    # Vertex of the parabola through the highest point and its two neighbours.
    k = int(np.clip(np.argmax(y), 1, len(y) - 2))
    coeffs = np.polyfit(t[k - 1:k + 2], y[k - 1:k + 2], 2)
    return float(-coeffs[1] / (2 * coeffs[0]))


refined = {size: parabola_peak(temps, column(size, "susceptibility")) for size in SIZES}
inverse_l = np.array([1 / s for s in SIZES])
slope, t_c_estimate = np.polyfit(inverse_l, [refined[s] for s in SIZES], 1)

# Seed check at L = 16: three more independent scans near the peak.
near = np.array([2.6, 2.55, 2.5, 2.46, 2.42, 2.38, 2.34, 2.3, 2.26, 2.22])
seed_peaks = [refined[16]]
for s in np.random.SeedSequence(9).spawn(3):
    scan = ising.temperature_scan(16, near, 1200, np.random.default_rng(s), burn_in=300)
    seed_peaks.append(parabola_peak(near, np.array([r.susceptibility for r in scan])))
seed_spread = float(np.std(seed_peaks, ddof=1))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(inverse_l, [refined[s] for s in SIZES], "o", color="#dc2626", ms=7,
        label="chi' peak")
x = np.linspace(0, 0.13, 50)
ax.plot(x, t_c_estimate + slope * x, "--", color="k", label="T_c + a/L")
ax.axhline(T_C, color="grey", ls=":", label="Onsager")
ax.set_xlabel("1 / L")
ax.set_ylabel("T_peak")
ax.legend()
plt.tight_layout()
plt.show()

for size in SIZES:
    print(f"L = {size:2d}: chi' peak at T = {refined[size]:.3f}")
print(f"extrapolated T_c = {t_c_estimate:.3f};  Onsager {T_C:.3f};  "
      f"difference {t_c_estimate - T_C:+.3f}")
print(f"L = 16 peak over four seeds: {np.round(seed_peaks, 3)}, spread {seed_spread:.3f}")

The quoted result is $T_c$ = (extrapolated value) $\pm$ (an uncertainty of the size of the
$L = 16$ seed spread, scaled up for an extrapolation that leans on three small lattices) — and two
caveats that no error bar captures: the extrapolation *assumes* $\nu = 1$, and the peak of
$\chi'$ is one of several finite-size definitions of "the transition temperature", each
drifting differently. What the measurement establishes is a trend that points at $2.27$; it does
not, and could not, show that anything is singular there. That is a statement about
$N \to \infty$, which no simulation reaches.

## Part 10 — Automated checks

A calculation you have not checked is a picture, not evidence. These assertions mirror the
project's test suite.

In [ ]:
# 1. Bookkeeping: the incrementally updated totals equal a from-scratch recount.
state = ising.random_state(16, rng)
_, _, state = ising.simulate(state, 2.3, 500, rng, burn_in=0)
assert state.energy == ising.lattice_energy(state.lattice)
assert state.magnetization == int(state.lattice.sum())

# 2. The chain reproduces its exact solution within four standard errors (Part 3).
for (temperature, field), results in chain.items():
    e_exact = ising.ising_1d_exact(temperature, field)[0]
    assert abs(results[:, 0].mean() - e_exact) < 4 * results[:, 0].std(ddof=1) / 2

# 3. Ordered below, disordered above; the deep-phase values match Onsager-Yang (Part 4).
for size in SIZES:
    m_abs = column(size, "abs_magnetization")
    assert m_abs[-1] > 0.95 and m_abs[0] < 0.35
assert relative_error(column(32, "abs_magnetization")[-1], ising.onsager_magnetization(1.8)) < 0.01

# 4. The susceptibility peak grows with L and sits above T_c, closer for larger L (Parts 4, 9).
heights = [column(size, "susceptibility").max() for size in SIZES]
assert heights[0] < heights[1] < heights[2]
assert refined[8] > refined[32] > T_C - 0.05

# 5. Critical slowing down: tau peaks near T_c and grows with L (Part 5).
for size in SIZES:
    tau = column(size, "tau_abs_magnetization")
    assert 2.15 < temps[np.argmax(tau)] < 2.6
assert tau_peak[8] < tau_peak[32]

# 6. The small lattice reverses, the large one does not (Part 6).
assert ising.branch_flips(branch[8]) >= 2 and ising.branch_flips(branch[32]) == 0

# 7. Below T_c the loop stays open; above it, it nearly closes (Part 7).
assert loops[(1.5, 40)][2] > 1.0 and loops[(3.0, 10)][2] < 0.3

# 8. Two rules, one equilibrium (Part 8).
assert rule_gap < 3 * rule_sigma

print("all checks passed")

## Part 11 — Explore it yourself

Choose the size, temperature, field, starting state, update rule and run length, then press
**Run Interact**. Left: the final configuration. Right: the $m$ and $e$ traces, with the
autocorrelation time of each. Things to try: a quench to $T = 1.0$ from a hot start at $L = 64$;
$T = 2.27$ at $L = 16$ and $L = 64$ with the same run length, comparing $\tau$; a small negative
field on an all-up lattice at $T = 1.5$; Glauber against Metropolis at $T = 2.27$.

In [ ]:
import ipywidgets as widgets


def explore(size=32, temperature=2.27, field=0.0, start="hot", rule="metropolis", sweeps=1000):
    gen = np.random.default_rng(0)
    state = (ising.random_state(size, gen, field=field) if start == "hot"
             else ising.with_field(ising.aligned_state(size), field))
    m, e, state = ising.simulate(state, temperature, sweeps, gen, burn_in=0, rule=rule)
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5),
                                      gridspec_kw={"width_ratios": [1, 1.6]})
    left.imshow(state.lattice, cmap="Blues", vmin=-1.5, vmax=1)
    left.set_xticks([])
    left.set_yticks([])
    left.set_title(f"L = {size}, T = {temperature}, h = {field}")
    right.plot(m, lw=0.8, label="m")
    right.plot(e, lw=0.8, label="E / (N J)")
    right.set_xlabel("sweep")
    right.legend()
    plt.tight_layout()
    plt.show()
    half = m[len(m) // 2:]
    print(f"last half of the run: <m> = {half.mean():+.3f}, <|m|> = {np.abs(half).mean():.3f}, "
          f"tau(|m|) = {ising.autocorrelation_time(np.abs(half)):.1f} sweeps, "
          f"tau(e) = {ising.autocorrelation_time(e[len(e) // 2:]):.1f} sweeps")
    print(f"Onsager-Yang |m| at this T: {ising.onsager_magnetization(temperature):.3f}")


widgets.interact_manual(
    explore,
    size=widgets.Dropdown(options=[16, 32, 64], value=32),
    temperature=widgets.FloatSlider(min=1.0, max=4.0, step=0.01, value=2.27),
    field=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0),
    start=widgets.ToggleButtons(options=["hot", "cold"]),
    rule=widgets.ToggleButtons(options=["metropolis", "glauber"]),
    sweeps=widgets.IntSlider(min=100, max=4000, step=100, value=1000),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "15-ising.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. In Part 6, both lattices were simulated from the same equations at the same temperature.
   Explain in your own words why one of them "chose" a direction and the other did not.
3. Your Part 9 measurement quotes a number with an uncertainty. List every assumption that
   number rests on that its error bar does *not* include.
4. Why can no simulation, however long, prove that the square lattice has a phase transition?

**Your answers:**

1.
2.
3.
4.